In [1]:

import os
import json
import time
import hashlib
import requests
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

# ── Paths ──────────────────────────────────────────────────────────────────

RAW_DIR   = Path("/mnt/data/raw")
META_FILE = Path("/mnt/data/metadata.jsonl")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── Semantic Scholar config ────────────────────────────────────────────────

S2_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY", "")
S2_HEADERS = {"x-api-key": S2_API_KEY} if S2_API_KEY else {}
S2_SEARCH  = "https://api.semanticscholar.org/graph/v1/paper/search"
S2_FIELDS  = "title,authors,year,externalIds,abstract,openAccessPdf,publicationTypes"

# ── Anchor papers corpus map ───────────────────────────────────────────────
# Format: (first_author, short_title, subfield, paper_type)
# paper_type: REVIEW | LANDMARK | BRIDGE

ANCHOR_PAPERS = [
    # 1. Value-based decision making
    ("Rangel",        "framework neurobiology value-based decision making",         "value_based_dm",        "REVIEW"),
    ("Padoa-Schioppa","neurobiology economic choice good-based model",              "value_based_dm",        "REVIEW"),
    ("Plassmann",     "orbitofrontal cortex encodes willingness to pay",            "value_based_dm",        "LANDMARK"),
    ("Padoa-Schioppa","neurons orbitofrontal cortex encode economic value",         "value_based_dm",        "LANDMARK"),
    ("Rustichini",    "neuro-computational model economic decisions",               "value_based_dm",        "BRIDGE"),

    # 2. Perceptual decision making
    ("Gold",          "neural basis decision making",                               "perceptual_dm",         "REVIEW"),
    ("Shadlen",       "neural basis perceptual decision parietal cortex",           "perceptual_dm",         "LANDMARK"),
    ("Britten",       "analysis visual motion neuronal psychophysical performance", "perceptual_dm",         "LANDMARK"),
    ("Summerfield",   "do humans make good decisions",                              "perceptual_dm",         "BRIDGE"),
    ("Drugowitsch",   "cost accumulating evidence perceptual decision making",      "perceptual_dm",         "BRIDGE"),

    # 3. Reinforcement learning in the brain
    ("Dayan",         "model-based model-free pavlovian reward learning",           "rl_brain",              "REVIEW"),
    ("Niv",           "reinforcement learning in the brain",                        "rl_brain",              "REVIEW"),
    ("Schultz",       "neural substrate prediction and reward",                     "rl_brain",              "LANDMARK"),
    ("Daw",           "cortical substrates exploratory decisions humans",           "rl_brain",              "LANDMARK"),
    ("Doll",          "ubiquity model-based reinforcement learning",                "rl_brain",              "BRIDGE"),

    # 4. Drift-diffusion / computational models
    ("Ratcliff",      "diffusion decision model theory data",                       "drift_diffusion",       "REVIEW"),
    ("Bogacz",        "physics optimal decision making",                            "drift_diffusion",       "REVIEW"),
    ("Krajbich",      "visual fixations computation value simple choice",           "drift_diffusion",       "BRIDGE"),
    ("Usher",         "time course perceptual choice leaky competing accumulator",  "drift_diffusion",       "LANDMARK"),
    ("Wiecki",        "HDDM hierarchical bayesian estimation diffusion decision",   "drift_diffusion",       "BRIDGE"),

    # 5. Neuroeconomics
    ("Camerer",       "neuroeconomics how neuroscience can inform economics",       "neuroeconomics",        "REVIEW"),
    ("Kahneman",      "prospect theory analysis decision under risk",               "neuroeconomics",        "LANDMARK"),
    ("Rangel",        "value normalization in decision making",                     "neuroeconomics",        "BRIDGE"),
    ("Fehr",          "social neuroeconomics neural circuitry social preferences",  "neuroeconomics",        "BRIDGE"),

    # 6. Cognitive control
    ("Badre",         "frontal cortex hierarchical control behavior",               "cognitive_control",     "REVIEW"),
    ("Shenhav",       "expected value of control integrative theory anterior cingulate", "cognitive_control","REVIEW"),
    ("Miller",        "integrative theory prefrontal cortex function",              "cognitive_control",     "LANDMARK"),
    ("Botvinick",     "motivation cognitive control behavior neural mechanism",     "cognitive_control",     "BRIDGE"),
    ("Daw",           "uncertainty-based competition prefrontal striatal systems",  "cognitive_control",     "BRIDGE"),

    # 7. Confidence and metacognition
    ("Fleming",       "neural basis metacognitive ability",                         "confidence_metacog",    "REVIEW"),
    ("Yeung",         "metacognition human decision-making",                        "confidence_metacog",    "REVIEW"),
    ("Kepecs",        "neural correlates computation decision confidence",          "confidence_metacog",    "LANDMARK"),
    ("Pouget",        "confidence and certainty distinct probabilistic quantities", "confidence_metacog",    "BRIDGE"),
    ("Donoso",        "foundations human reasoning prefrontal cortex",              "confidence_metacog",    "BRIDGE"),

    # 8. SC and subcortical decision making
    ("Wurtz",         "visual-motor function primate superior colliculus",          "sc_subcortical",        "REVIEW"),
    ("Felsen",        "neural substrates sensory-guided locomotor decisions superior colliculus", "sc_subcortical", "REVIEW"),
    ("Basso",         "modulation neuronal activity superior colliculus target probability",      "sc_subcortical", "LANDMARK"),
    ("Jun",           "causal role primate superior colliculus computation evidence perceptual",  "sc_subcortical", "LANDMARK"),
    ("Crapse",        "role superior colliculus decision criteria",                 "sc_subcortical",        "LANDMARK"),
    ("Krauzlis",      "superior colliculus visual spatial attention",               "sc_subcortical",        "BRIDGE"),
    ("Felsen",        "integrative role superior colliculus selecting targets movements",         "sc_subcortical", "BRIDGE"),
    ("Duan",          "primate superior colliculus causally engaged abstract higher-order cognition", "sc_subcortical", "BRIDGE"),
    ("McAlonan",      "superior colliculus inactivation biases target choice",      "sc_subcortical",        "BRIDGE"),
    ("Kim",           "superior colliculus encodes alternations spatial target selections",       "sc_subcortical", "BRIDGE"),
]


In [4]:
import requests

paperId = "649def34f8be52c8b66281af98ae884c09aef38b"

# Define the API endpoint URL
url = f"http://api.semanticscholar.org/graph/v1/paper/{paperId}"

# Define the query parameters
query_params = {"fields": "title,year,abstract,citationCount"}

# Directly define the API key (Reminder: Securely handle API keys in production environments)
api_key = "your api key goes here"  # Replace with the actual API key

# Define headers with API key
headers = {"x-api-key": api_key}

# Send the API request
response = requests.get(url, params=query_params, headers=headers)

# Check response status
if response.status_code == 200:
   response_data = response.json()
   # Process and print the response data as needed
   print(response_data)
else:
   print(f"Request failed with status code {response.status_code}: {response.text}")

Request failed with status code 403: {"message":"Forbidden"}


### Unpaywall

In [ ]:
# test Unpaywall right now
curl "https://api.unpaywall.org/v2/10.1038/nn.4270?email=vaibhavt459@gmail.com"

In [8]:
"""
fetch_papers.py — Cortexa anchor paper fetcher

Source priority for each paper:
  1. PubMed E-utilities  — metadata + abstract + DOI + PMCID
  2. Unpaywall           — open-access PDF lookup by DOI
  3. PubMed Central      — free full text if PMCID exists
  4. Sci-Hub             — last resort fallback

Usage (inside container):
    python src/fetch_papers.py

Outputs:
    /mnt/data/raw/           PDFs where available
    /mnt/data/metadata.jsonl one record per paper
"""

import json
import time
import hashlib
import requests
import re
from pathlib import Path

# ── Config ─────────────────────────────────────────────────────────────────

RAW_DIR   = Path("/mnt/data/raw")
META_FILE = Path("/mnt/data/metadata.jsonl")
RAW_DIR.mkdir(parents=True, exist_ok=True)

UNPAYWALL_EMAIL = "vaibhavt459@gmail.com"
PUBMED_BASE     = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
UNPAYWALL_BASE  = "https://api.unpaywall.org/v2"
PMC_BASE        = "https://www.ncbi.nlm.nih.gov/pmc/articles"
SCIHUB_BASE     = "https://sci-hub.ru"

HEADERS = {"User-Agent": "Cortexa/0.1 (research tool; mailto:vaibhavt459@gmail.com)"}

# ── Anchor papers ──────────────────────────────────────────────────────────

ANCHOR_PAPERS = [
    # 1. Value-based decision making
    ("framework neurobiology value-based decision making",                       "Rangel",         2008, "value_based_dm",     "REVIEW"),
    ("neurobiology economic choice good-based model",                            "Padoa-Schioppa", 2011, "value_based_dm",     "REVIEW"),
    ("orbitofrontal cortex encodes willingness to pay",                          "Plassmann",      2007, "value_based_dm",     "LANDMARK"),
    ("neurons orbitofrontal cortex encode economic value",                       "Padoa-Schioppa", 2006, "value_based_dm",     "LANDMARK"),
    ("neuro-computational model economic decisions",                             "Rustichini",     2015, "value_based_dm",     "BRIDGE"),

    # 2. Perceptual decision making
    ("neural basis decision making annual review neuroscience",                  "Gold",           2007, "perceptual_dm",      "REVIEW"),
    ("neural basis perceptual decision lateral intraparietal",                   "Shadlen",        2001, "perceptual_dm",      "LANDMARK"),
    ("analysis visual motion neuronal psychophysical performance",               "Britten",        1992, "perceptual_dm",      "LANDMARK"),
    ("do humans make good decisions",                                            "Summerfield",    2015, "perceptual_dm",      "BRIDGE"),
    ("cost accumulating evidence perceptual decision making",                    "Drugowitsch",    2012, "perceptual_dm",      "BRIDGE"),

    # 3. Reinforcement learning in the brain
    ("model-based model-free pavlovian reward learning",                         "Dayan",          2014, "rl_brain",           "REVIEW"),
    ("reinforcement learning in the brain",                                      "Niv",            2009, "rl_brain",           "REVIEW"),
    ("neural substrate prediction and reward",                                   "Schultz",        1997, "rl_brain",           "LANDMARK"),
    ("cortical substrates exploratory decisions humans",                         "Daw",            2006, "rl_brain",           "LANDMARK"),
    ("ubiquity model-based reinforcement learning",                              "Doll",           2012, "rl_brain",           "BRIDGE"),

    # 4. Drift-diffusion / computational models
    ("diffusion decision model theory data",                                     "Ratcliff",       2008, "drift_diffusion",    "REVIEW"),
    ("physics optimal decision making",                                          "Bogacz",         2006, "drift_diffusion",    "REVIEW"),
    ("visual fixations computation value simple choice",                         "Krajbich",       2010, "drift_diffusion",    "BRIDGE"),
    ("time course perceptual choice leaky competing accumulator",                "Usher",          2001, "drift_diffusion",    "LANDMARK"),
    ("HDDM hierarchical bayesian estimation diffusion decision",                 "Wiecki",         2013, "drift_diffusion",    "BRIDGE"),

    # 5. Neuroeconomics
    ("neuroeconomics how neuroscience can inform economics",                     "Camerer",        2005, "neuroeconomics",     "REVIEW"),
    ("prospect theory analysis decision under risk",                             "Kahneman",       1979, "neuroeconomics",     "LANDMARK"),
    ("value normalization in decision making",                                   "Rangel",         2012, "neuroeconomics",     "BRIDGE"),
    ("social neuroeconomics neural circuitry social preferences",                "Fehr",           2007, "neuroeconomics",     "BRIDGE"),

    # 6. Cognitive control
    ("frontal cortex hierarchical control behavior",                             "Badre",          2018, "cognitive_control",  "REVIEW"),
    ("expected value of control anterior cingulate",                             "Shenhav",        2013, "cognitive_control",  "REVIEW"),
    ("integrative theory prefrontal cortex function",                            "Miller",         2001, "cognitive_control",  "LANDMARK"),
    ("motivation cognitive control behavior neural mechanism",                   "Botvinick",      2015, "cognitive_control",  "BRIDGE"),
    ("uncertainty-based competition prefrontal striatal systems",                "Daw",            2005, "cognitive_control",  "BRIDGE"),

    # 7. Confidence and metacognition
    ("neural basis metacognitive ability",                                       "Fleming",        2012, "confidence_metacog", "REVIEW"),
    ("metacognition human decision-making",                                      "Yeung",          2012, "confidence_metacog", "REVIEW"),
    ("neural correlates computation decision confidence",                        "Kepecs",         2008, "confidence_metacog", "LANDMARK"),
    ("confidence and certainty distinct probabilistic quantities",               "Pouget",         2016, "confidence_metacog", "BRIDGE"),
    ("foundations human reasoning prefrontal cortex",                           "Donoso",         2014, "confidence_metacog", "BRIDGE"),

    # 8. SC and subcortical decision making
    ("visual-motor function primate superior colliculus",                        "Wurtz",          1980, "sc_subcortical",     "REVIEW"),
    ("neural substrates sensory-guided locomotor decisions superior colliculus", "Felsen",         2008, "sc_subcortical",     "REVIEW"),
    ("modulation neuronal activity superior colliculus target probability",      "Basso",          1998, "sc_subcortical",     "LANDMARK"),
    ("causal role primate superior colliculus computation evidence perceptual",  "Jun",            2021, "sc_subcortical",     "LANDMARK"),
    ("role superior colliculus decision criteria",                               "Crapse",         2018, "sc_subcortical",     "LANDMARK"),
    ("superior colliculus visual spatial attention",                             "Krauzlis",       2013, "sc_subcortical",     "BRIDGE"),
    ("integrative role superior colliculus selecting targets movements",         "Felsen",         2012, "sc_subcortical",     "BRIDGE"),
    ("primate superior colliculus causally engaged abstract higher-order",       "Duan",           2024, "sc_subcortical",     "BRIDGE"),
    ("superior colliculus inactivation biases target choice mice",               "McAlonan",       2020, "sc_subcortical",     "BRIDGE"),
    ("superior colliculus encodes alternations spatial target selections",       "Kim",            2008, "sc_subcortical",     "BRIDGE"),
]

# ── Helpers ────────────────────────────────────────────────────────────────

def title_slug(title: str) -> str:
    return hashlib.md5(title.lower().encode()).hexdigest()[:12]


def load_seen() -> set:
    seen = set()
    if META_FILE.exists():
        for line in META_FILE.read_text().splitlines():
            try:
                r = json.loads(line)
                seen.add(f"{r['first_author'].lower()}_{r['year']}")
            except Exception:
                pass
    return seen


def save_meta(record: dict):
    with open(META_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")


def download_pdf(url: str, dest: Path) -> bool:
    """Download a PDF from url to dest. Returns True if successful."""
    try:
        r = requests.get(url, timeout=30, headers=HEADERS, allow_redirects=True)
        if r.status_code == 200 and len(r.content) > 5_000:
            if b"%PDF" in r.content[:8]:
                dest.write_bytes(r.content)
                return True
            else:
                print(f"    Not a valid PDF (missing header)")
    except Exception as e:
        print(f"    Download error: {e}")
    return False


# ── Source 1: PubMed ───────────────────────────────────────────────────────

def pubmed_fetch(title: str, author: str, year: int) -> dict | None:
    """Search PubMed, return metadata dict with doi, pmid, pmcid, abstract."""
    query = f"{title} {author}[author]"
    try:
        # search with year filter first
        for params in [
            {"db": "pubmed", "term": query, "retmax": 3, "retmode": "json",
             "datetype": "pdat", "mindate": str(year-1), "maxdate": str(year+1)},
            {"db": "pubmed", "term": query, "retmax": 3, "retmode": "json"},
        ]:
            r = requests.get(f"{PUBMED_BASE}/esearch.fcgi", params=params, timeout=10)
            ids = r.json().get("esearchresult", {}).get("idlist", [])
            if ids:
                break

        if not ids:
            return None

        time.sleep(0.4)

        # fetch summary for DOI and PMCID
        r = requests.get(f"{PUBMED_BASE}/esummary.fcgi",
                         params={"db": "pubmed", "id": ids[0], "retmode": "json"},
                         timeout=10)
        s = r.json().get("result", {}).get(ids[0], {})

        doi   = next((x["value"] for x in s.get("articleids", []) if x["idtype"] == "doi"),  None)
        pmcid = next((x["value"] for x in s.get("articleids", []) if x["idtype"] == "pmc"),  None)

        time.sleep(0.4)

        # fetch abstract text
        r = requests.get(f"{PUBMED_BASE}/efetch.fcgi",
                         params={"db": "pubmed", "id": ids[0],
                                 "rettype": "abstract", "retmode": "text"},
                         timeout=10)
        abstract = r.text.strip()

        return {
            "pmid":    ids[0],
            "pmcid":   pmcid,
            "doi":     doi,
            "title":   s.get("title", ""),
            "authors": [a["name"] for a in s.get("authors", [])],
            "year":    s.get("pubdate", str(year))[:4],
            "journal": s.get("source", ""),
            "abstract": abstract,
        }

    except Exception as e:
        print(f"    PubMed error: {e}")
        return None


# ── Source 2: Unpaywall ────────────────────────────────────────────────────

def unpaywall_fetch(doi: str) -> str | None:
    """Return best open-access PDF URL for a DOI, or None."""
    if not doi:
        return None
    try:
        r = requests.get(f"{UNPAYWALL_BASE}/{doi}",
                         params={"email": UNPAYWALL_EMAIL}, timeout=10)
        if r.status_code != 200:
            return None
        data = r.json()
        best = data.get("best_oa_location")
        if best and best.get("url_for_pdf"):
            return best["url_for_pdf"]
        for loc in data.get("oa_locations", []):
            if loc.get("url_for_pdf"):
                return loc["url_for_pdf"]
        return None
    except Exception as e:
        print(f"    Unpaywall error: {e}")
        return None


# ── Source 3: PubMed Central ───────────────────────────────────────────────

def pmc_fetch(pmcid: str) -> str | None:
    """Return PDF URL from PubMed Central OA API if paper is open access."""
    if not pmcid:
        return None
    pmcid_clean = pmcid if pmcid.startswith("PMC") else f"PMC{pmcid}"
    try:
        # use OA API to check if open access and get direct PDF link
        r = requests.get(
            "https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi",
            params={"id": pmcid_clean}, timeout=10,
        )
        if r.status_code != 200:
            return None
        # look for PDF link in XML response
        # format: <link format="pdf" href="ftp://..."/>
        pdf_match = re.search(r'href="(ftp://[^"]+\.pdf)"', r.text)
        if pdf_match:
            ftp_url = pdf_match.group(1)
            # convert ftp to https for requests
            https_url = ftp_url.replace("ftp://ftp.ncbi.nlm.nih.gov", "https://ftp.ncbi.nlm.nih.gov")
            return https_url
        # also check for https links
        https_match = re.search(r'href="(https://[^"]+\.pdf)"', r.text)
        if https_match:
            return https_match.group(1)
        return None
    except Exception as e:
        print(f"    PMC error: {e}")
        return None


# ── Source 4: Sci-Hub ──────────────────────────────────────────────────────

def scihub_fetch(doi: str) -> str | None:
    """Return direct PDF URL from Sci-Hub for a DOI, or None."""
    if not doi:
        return None
    try:
        url = f"{SCIHUB_BASE}/{doi}"
        r = requests.get(url, timeout=15, headers=HEADERS)
        if r.status_code != 200:
            return None
        # parse PDF embed/iframe src from response HTML
        patterns = [
            rb'<iframe[^>]+src=["\']([^"\']+\.pdf[^"\']*)["\']',
            rb'<embed[^>]+src=["\']([^"\']+\.pdf[^"\']*)["\']',
            rb'location\.href\s*=\s*["\']([^"\']+\.pdf[^"\']*)["\']',
            rb'href=["\']([^"\']+\.pdf[^"\']*)["\']',
        ]
        for pattern in patterns:
            match = re.search(pattern, r.content)
            if match:
                pdf_url = match.group(1).decode().strip()
                if pdf_url.startswith("//"):
                    pdf_url = "https:" + pdf_url
                elif pdf_url.startswith("/"):
                    pdf_url = SCIHUB_BASE + pdf_url
                return pdf_url
        return None
    except Exception as e:
        print(f"    Sci-Hub error: {e}")
        return None


# ── Main fetch loop ────────────────────────────────────────────────────────

def try_download(label: str, url_fn, arg: str, dest: Path) -> str | None:
    """Try to get a PDF URL and download it. Returns path string or None."""
    url = url_fn(arg)
    if not url:
        print(f"    {label}: no PDF found")
        return None
    print(f"    {label}: found PDF, downloading...")
    if download_pdf(url, dest):
        print(f"    {label}: saved {dest.name}")
        return str(dest)
    print(f"    {label}: download failed")
    return None


def fetch_all():
    seen = load_seen()
    print(f"Already in corpus: {len(seen)} papers")
    print(f"Papers to fetch:   {len(ANCHOR_PAPERS)}")
    print(f"Sources:           PubMed → Unpaywall → PMC → Sci-Hub\n")

    stats = {"fetched": 0, "pdf": 0, "abstract_only": 0,
             "no_pubmed": 0, "skipped": 0}

    for i, (title, author, year, subfield, paper_type) in enumerate(ANCHOR_PAPERS):
        key = f"{author.lower()}_{year}"
        print(f"\n[{i+1:02d}/{len(ANCHOR_PAPERS)}] [{paper_type}] {author} {year}")
        print(f"  {title[:65]}")

        if key in seen:
            print(f"  Already fetched — skipping")
            stats["skipped"] += 1
            continue

        # ── Step 1: PubMed metadata ────────────────────────────────────────
        print(f"  [1] PubMed...")
        pm = pubmed_fetch(title, author, year)

        if pm is None:
            print(f"  Not found on PubMed — saving stub")
            save_meta({
                "title": title, "first_author": author, "year": str(year),
                "subfield": subfield, "paper_type": paper_type,
                "source": "not_found", "has_full_text": False,
                "doi": None, "pmid": None, "pmcid": None,
                "abstract": "", "pdf_path": None,
            })
            seen.add(key)
            stats["no_pubmed"] += 1
            time.sleep(1)
            continue

        print(f"  Found: {pm['title'][:65]}")
        doi   = pm.get("doi")
        pmcid = pm.get("pmcid")
        print(f"  DOI: {doi or 'none'}  |  PMCID: {pmcid or 'none'}")

        # ── Steps 2-4: PDF acquisition ─────────────────────────────────────
        pdf_path = None
        slug = title_slug(pm["title"])

        # 2. Unpaywall
        print(f"  [2] Unpaywall...")
        pdf_path = try_download("Unpaywall", unpaywall_fetch, doi, RAW_DIR / f"{slug}.pdf")

        # 3. PubMed Central
        if not pdf_path and pmcid:
            print(f"  [3] PubMed Central...")
            pdf_path = try_download("PMC", pmc_fetch, pmcid, RAW_DIR / f"{slug}_pmc.pdf")
        elif not pdf_path:
            print(f"  [3] PMC: no PMCID available")

        # 4. Sci-Hub
        if not pdf_path and doi:
            print(f"  [4] Sci-Hub...")
            pdf_path = try_download("Sci-Hub", scihub_fetch, doi, RAW_DIR / f"{slug}_sh.pdf")
        elif not pdf_path:
            print(f"  [4] Sci-Hub: no DOI available")

        if not pdf_path:
            print(f"  All sources exhausted — abstract only")

        # ── Save record ────────────────────────────────────────────────────
        record = {
            "title":         pm["title"],
            "first_author":  author,
            "authors":       pm["authors"],
            "year":          pm["year"],
            "doi":           doi,
            "pmid":          pm["pmid"],
            "pmcid":         pmcid,
            "abstract":      pm["abstract"],
            "journal":       pm["journal"],
            "subfield":      subfield,
            "paper_type":    paper_type,
            "source":        "pubmed+unpaywall+pmc+scihub",
            "pdf_path":      pdf_path,
            "has_full_text": pdf_path is not None,
        }
        save_meta(record)
        seen.add(key)

        stats["fetched"] += 1
        if pdf_path:
            stats["pdf"] += 1
        else:
            stats["abstract_only"] += 1

        time.sleep(1.0)

    print(f"\n{'─'*55}")
    print(f"Done.")
    print(f"  Fetched:       {stats['fetched']}")
    print(f"  Full PDF:      {stats['pdf']}")
    print(f"  Abstract only: {stats['abstract_only']}")
    print(f"  Not on PubMed: {stats['no_pubmed']}")
    print(f"  Skipped:       {stats['skipped']}")
    print(f"\n  Metadata → {META_FILE}")
    print(f"  PDFs     → {RAW_DIR}")
    print(f"\nNext step: python src/ingest.py")


if __name__ == "__main__":
    fetch_all()

Already in corpus: 0 papers
Papers to fetch:   44
Sources:           PubMed → Unpaywall → PMC → Sci-Hub


[01/44] [REVIEW] Rangel 2008
  framework neurobiology value-based decision making
  [1] PubMed...
  Found: A framework for studying the neurobiology of value-based decision
  DOI: 10.1038/nrn2357  |  PMCID: PMC4332708
  [2] Unpaywall...
    Unpaywall: no PDF found
  [3] PubMed Central...
    PMC: no PDF found
  [4] Sci-Hub...
    Sci-Hub: no PDF found
  All sources exhausted — abstract only

[02/44] [REVIEW] Padoa-Schioppa 2011
  neurobiology economic choice good-based model
  [1] PubMed...
  Found: Neurobiology of economic choice: a good-based model.
  DOI: 10.1146/annurev-neuro-061010-113648  |  PMCID: PMC3273993
  [2] Unpaywall...
    Unpaywall: no PDF found
  [3] PubMed Central...
    PMC: no PDF found
  [4] Sci-Hub...
    Sci-Hub: no PDF found
  All sources exhausted — abstract only

[03/44] [LANDMARK] Plassmann 2007
  orbitofrontal cortex encodes willingness to pay
  [1] PubM

In [2]:
RAW_DIR

PosixPath('/mnt/data/raw')

# Load All Data

In [18]:
landmark_pdf_paths = list(RAW_DIR.glob("*.pdf"))
landmark_metadata = RAW_DIR / "metadata.jsonl"
keyword_metadata = RAW_DIR / "kw_metadata.jsonl"

perceptual_pdf_paths = list(Path(RAW_DIR, "old_folders", "perceptual_decision_making", "pdf").glob("*.pdf"))
perceptual_metadata = Path(RAW_DIR, "old_folders", "perceptual_decision_making", "metadata.pkl")


sc_pdf_paths = list(Path(RAW_DIR, "old_folders", "superior_colliculus", "pdf").glob("*.pdf"))
sc_metadata = Path(RAW_DIR, "old_folders", "superior_colliculus", "metadata.pkl")

In [32]:
# load all four metadata files into dicts for easy access during ingestion

def load_metadata(path: Path) -> list[dict]:
    records = []
    with open(path) as f:
        for line in f:
            try:
                records.append(json.loads(line))
            except Exception:
                pass
    return records

def load_pickle(path: Path) -> list[dict]:
    import pickle
    with open(path, "rb") as f:

        return pickle.load(f).to_dict(orient="records")


landmark_metadata_dict = load_metadata(landmark_metadata)
perceptual_metadata_dict = load_pickle(perceptual_metadata)
sc_metadata_dict = load_pickle(sc_metadata)
keyword_metadata_dict = load_metadata(keyword_metadata)


In [25]:
landmark_metadata_dict[0]

{'title': 'A framework for studying the neurobiology of value-based decision making.',
 'first_author': 'Rangel',
 'authors': ['Rangel A', 'Camerer C', 'Montague PR'],
 'year': '2008',
 'doi': '10.1038/nrn2357',
 'pmid': '18545266',
 'pmcid': 'PMC4332708',
 'abstract': '1. Nat Rev Neurosci. 2008 Jul;9(7):545-56. doi: 10.1038/nrn2357. Epub 2008 Jun\n11.\n\nA framework for studying the neurobiology of value-based decision making.\n\nRangel A(1), Camerer C, Montague PR.\n\nAuthor information:\n(1)Division of the Humanities and Social Sciences (HSS) and Computational and \nNeural Systems Program, California Institute of Technology, Pasadena, California \n91125, USA. rangel@hss.caltech.edu\n\nNeuroeconomics is the study of the neurobiological and computational basis of \nvalue-based decision making. Its goal is to provide a biologically based account \nof human behaviour that can be applied in both the natural and the social \nsciences. This Review proposes a framework to investigate differ

In [26]:
keyword_metadata_dict[0]

{'title': 'An Iteratively Weighted MMSE Approach to Distributed Sum-Utility Maximization for a MIMO Interfering Broadcast Channel',
 'authors': ['Qingjiang Shi', 'Meisam Razaviyayn', 'Z. Luo', 'Chen He'],
 'year': 2011,
 'doi': '10.1109/TSP.2011.2147784',
 'arxiv_id': '',
 'venue': 'IEEE Transactions on Signal Processing',
 'citations': 2091,
 'abstract': None,
 'oa_url': '',
 'subfield': 'value_based_dm',
 'query': 'utility maximization prefrontal',
 'source': 'keyword_search',
 'has_full_text': True,
 'pdf_path': '/mnt/data/raw/4b0b6c9dd806_sh.pdf'}

In [33]:
sc_metadata_dict[0]

{'paperId': '000c8597045ae34ebed161c2dc259d142cf181bb',
 'url': 'https://www.semanticscholar.org/paper/000c8597045ae34ebed161c2dc259d142cf181bb',
 'title': 'Monosynaptic trans-collicular pathways for sensory-motor integration in the whisker system',
 'abstract': 'The superior colliculus (SC), a conserved midbrain-node with extensive long-range connectivity throughout the brain, is a key structure for innate behaviors. Descending cortical pathways are increasingly recognized as central control points for SC-mediated behaviors, but how cortico-collicular pathways coordinate SC activity at the cellular level is poorly understood. Moreover, despite the known role of the SC as a multisensory integrator, the involvement of the SC in the somatosensory system is largely unexplored in comparison to its involvement in the visual and auditory systems. Here, we mapped the connectivity of the whisker-sensitive region of the SC in mice with trans-synaptic and intersectional tracing tools and in vivo

In [34]:
perceptual_metadata_dict[0]

{'paperId': '000c8597045ae34ebed161c2dc259d142cf181bb',
 'url': 'https://www.semanticscholar.org/paper/000c8597045ae34ebed161c2dc259d142cf181bb',
 'title': 'Monosynaptic trans-collicular pathways for sensory-motor integration in the whisker system',
 'abstract': 'The superior colliculus (SC), a conserved midbrain-node with extensive long-range connectivity throughout the brain, is a key structure for innate behaviors. Descending cortical pathways are increasingly recognized as central control points for SC-mediated behaviors, but how cortico-collicular pathways coordinate SC activity at the cellular level is poorly understood. Moreover, despite the known role of the SC as a multisensory integrator, the involvement of the SC in the somatosensory system is largely unexplored in comparison to its involvement in the visual and auditory systems. Here, we mapped the connectivity of the whisker-sensitive region of the SC in mice with trans-synaptic and intersectional tracing tools and in vivo